# Study Bounce — Telescope-position bounce tests (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-05-06  
**Last Modified:** 2026-05-06  
**Status:** Draft  
**Keywords:** AOS, FAM, bounce test, BLOCK-T720, BLOCK-T724, double Zernike

## Description

BLOCK-T720 and BLOCK-T724 are *bounce* tests: take a FAM triplet at one
telescope orientation, move to a second orientation, take another
triplet, and repeat — alternating back and forth.

* **T720** bounces between Elevation = 70° and Elevation = 40°
  (Rotator near 0° throughout).
* **T724** bounces between Rotator = 0° and Rotator = 60°
  (Elevation near 70° throughout).

This notebook reads one or more `*_fits.parquet` files, identifies the
visits in each *setting* of each bounce, computes the **median** and
**robust RMS** (`1.4826 × MAD`) for every Double-Zernike coefficient,
and builds the difference (comparison − reference) per (k, j) along
with its propagated uncertainty.  The summary plot is a heat-map over
(k, j) showing ΔDZ and its 1-σ error in each cell.

**Output:** PDF of bounce summary heatmaps + long-format parquet of
per-(k, j) statistics.  Files in `output/study_bounce/`.

**Based on:** the per-(k, j) tracking conventions used in the
`fit_params_resid_*` plots and `study_50dofLUT.ipynb`.  Designed to be
extended to more bounce tests with different telescope positions.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-05-06 | Aaron Roodman | Initial version: T720 (Elev 70/40) + T724 (Rot 0/60) heatmap summary. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Bounce Analysis](#analysis)
6. [Summary Heatmaps](#plots)
7. [Output Tables](#output)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input fit parquets (one or more) -----
# Multiple files are concatenated row-wise.  Bad-flagged visits are
# dropped before any statistics are computed.
fits_parquet_paths = [
    'output/aos_fam_danish_v1_triplets_bin_2x_20260418_20260430_fits.parquet',
]

# Fit prefix.
fit_prefix = 'z1toz6'

# Pupil-Noll j list.  None -> auto from the `nollIndices` column on
# the first fits parquet.
pupil_j_range = None
# Focal-plane Noll k list.
focal_k_range = range(1, 7)

# ----- Bounce specifications -----
# Each bounce has a `program` (a science_program value or list of
# values to filter to), a `reference` setting, and one or more
# `comparisons`.  A *setting* is a dict of selection criteria; only
# the keys that are present are applied.  Allowed keys (degrees /
# day_obs ints):
#     alt_range       -> (lo, hi) elevation range
#     rotator_range   -> (lo, hi)
#     day_obs_range   -> (lo, hi)
#     seq_num_range   -> (lo, hi)
#     mask            -> arbitrary callable: f(fit_table) -> bool array
#
# `program` is applied to every setting in that bounce so both the
# reference and the comparison only pull from the matching block —
# fixes the case where alt/rot windows accidentally include visits
# from BLOCK-T614 or BLOCK-T710_v2.
#
# The `delta` heatmap shows comparison.median - reference.median per
# (k, j), with errors from the per-setting MAD-based standard error of
# the median added in quadrature.
bounces = [
    {
        'name': 'T720_elevation',
        'description': 'Elevation 40 − 70 deg, rotator ≈ 0',
        'program': 'BLOCK-T720',
        'reference': {
            'label': 'Elev=70',
            'alt_range':     (67.0, 73.0),
            'rotator_range': (-3.0,  3.0),
        },
        'comparisons': [
            {
                'label': 'Elev=40',
                'alt_range':     (37.0, 43.0),
                'rotator_range': (-3.0,  3.0),
            },
        ],
    },
    {
        'name': 'T724_rotator',
        'description': 'Rotator 60 − 0 deg, elevation ≈ 70',
        'program': 'BLOCK-T724',
        'reference': {
            'label': 'Rot=0',
            'rotator_range': (-3.0,  3.0),
            'alt_range':     (67.0, 73.0),
        },
        'comparisons': [
            {
                'label': 'Rot=60',
                'rotator_range': (57.0, 63.0),
                'alt_range':     (67.0, 73.0),
            },
        ],
    },
]

# ----- Heatmap styling -----
# Color scale: per-bounce 95th-percentile of |Δ| if `vlim_um` is None,
# else use the explicit ±vlim_um for every heatmap.  Diverging colormap.
heatmap_vlim_um   = None
heatmap_cell_fontsize = 7
# Significance heatmap: Delta / err, capped at ±sig_vlim.
sig_vlim          = 5.0

# Pass/fail summary heatmap thresholds.  A (k, j) cell
# is marked as a *real* shift if BOTH
#     |Δ DZ_kj|    > pass_delta_threshold_um   (μm), AND
#     |Δ / σ|      > pass_nsigma_threshold     (unitless).
pass_nsigma_threshold   = 3.5
pass_delta_threshold_um = 0.01

# ----- Output -----
output_dir         = 'output/study_bounce'
output_pdf         = f'{output_dir}/study_bounce_summary.pdf'
output_table_path  = f'{output_dir}/study_bounce_kj_stats.parquet'

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable, vstack

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

print('Ready.')

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
from common.zernike_names import (
    NOLL_NAMES, NOLL_FORMULAS, FOCAL_NAMES, PUPIL_NAMES,
)
def _alt_to_deg(alt_arr):
    """Auto-detect radians vs degrees in `alt`."""
    a = np.asarray(alt_arr, dtype=float)
    if np.nanmax(np.abs(a)) < 2.0 * np.pi + 1e-3:
        return np.rad2deg(a)
    return a


def filter_visits(fit_table, *, alt_range=None, rotator_range=None,
                  day_obs_range=None, seq_num_range=None,
                  program=None, mask=None):
    """Build a boolean mask for visits matching every supplied criterion.

    All ranges are inclusive.  Missing keys are unrestricted.

    `program` matches the `science_program` column (case-sensitive
    exact match).  Pass a single string or a list/tuple of strings;
    the result is the union of all matches.  The fits parquet only
    started carrying `science_program` after the upstream
    `intrinsics_lib.merge_program_reason_to_visit_info` change — when
    the column is missing the program filter is silently skipped and a
    one-time warning is printed.
    """
    n = len(fit_table)
    keep = np.ones(n, dtype=bool)
    if alt_range is not None and 'alt' in fit_table.colnames:
        alt_deg = _alt_to_deg(fit_table['alt'])
        keep &= (alt_deg >= alt_range[0]) & (alt_deg <= alt_range[1])
    if rotator_range is not None and 'rotator_angle' in fit_table.colnames:
        rot = np.asarray(fit_table['rotator_angle'], dtype=float)
        keep &= (rot >= rotator_range[0]) & (rot <= rotator_range[1])
    if day_obs_range is not None and 'day_obs' in fit_table.colnames:
        d = np.asarray(fit_table['day_obs']).astype(int)
        keep &= (d >= int(day_obs_range[0])) & (d <= int(day_obs_range[1]))
    if seq_num_range is not None and 'seq_num' in fit_table.colnames:
        s = np.asarray(fit_table['seq_num']).astype(int)
        keep &= (s >= int(seq_num_range[0])) & (s <= int(seq_num_range[1]))
    if program is not None:
        if 'science_program' not in fit_table.colnames:
            if not getattr(filter_visits, '_warned_missing_program', False):
                print("  WARNING: 'science_program' not in fit_table — "
                      "the program filter is being ignored.  Re-run mktable "
                      "with the updated intrinsics_lib to populate it.")
                filter_visits._warned_missing_program = True
        else:
            sp = np.asarray(fit_table['science_program']).astype(str)
            if isinstance(program, (list, tuple, set)):
                prog_set = {str(p) for p in program}
                keep &= np.array([s in prog_set for s in sp])
            else:
                keep &= (sp == str(program))
    if mask is not None:
        keep &= np.asarray(mask, dtype=bool)
    return keep


def visit_list_str(fit_table, mask, max_per_day=20):
    """Pretty-print summary of which (day_obs, seq_num) visits survive a mask.

    Returns a string with one line per day_obs.  Truncates the seq_num
    list if it has more than `max_per_day` entries.
    """
    if not np.any(mask):
        return '    (no visits)'
    sub = fit_table[mask]
    dobs = np.asarray(sub['day_obs']).astype(int)
    snum = np.asarray(sub['seq_num']).astype(int)
    order = np.lexsort((snum, dobs))
    dobs = dobs[order]; snum = snum[order]
    lines = []
    for d in sorted(set(dobs.tolist())):
        s_list = sorted(snum[dobs == d].tolist())
        if len(s_list) > max_per_day:
            shown = (', '.join(str(x) for x in s_list[:max_per_day // 2])
                     + ', …, '
                     + ', '.join(str(x) for x in s_list[-max_per_day // 2:]))
            lines.append(f'    {d}: {len(s_list):3d} seq_num — '
                         f'[{shown}]')
        else:
            lines.append(f'    {d}: {len(s_list):3d} seq_num — '
                         f'{s_list}')
    return '\n'.join(lines)


def stats_per_kj(fit_table, mask, prefix, k_range, j_range):
    """Per-(k, j) median, robust RMS (1.4826*MAD), SEM of the median, n.

    SEM (standard error of the median) for normally distributed values
    is `1.2533 * sigma / sqrt(n)`; we use the MAD-based sigma estimate.
    """
    sub = fit_table[mask]
    out = {}
    for j in j_range:
        for k in k_range:
            col = f'{prefix}_z{j}_c{k}'
            if col not in sub.colnames:
                continue
            vals = np.asarray(sub[col], dtype=float)
            vals = vals[np.isfinite(vals)]
            n = int(len(vals))
            if n < 3:
                out[(int(k), int(j))] = {
                    'median': np.nan, 'sigma_mad': np.nan,
                    'sem': np.nan, 'n': n}
                continue
            med = float(np.median(vals))
            mad = float(np.median(np.abs(vals - med)))
            sigma_mad = 1.4826 * mad
            sem = 1.2533 * sigma_mad / np.sqrt(n)
            out[(int(k), int(j))] = {
                'median': med, 'sigma_mad': sigma_mad,
                'sem': sem, 'n': n}
    return out


def diff_stats(stats_comp, stats_ref):
    """Difference (comparison - reference) per (k, j) with quadrature errors."""
    out = {}
    keys = set(stats_comp.keys()) & set(stats_ref.keys())
    for kj in keys:
        a = stats_comp[kj]; b = stats_ref[kj]
        if not (np.isfinite(a['median']) and np.isfinite(b['median'])):
            out[kj] = {'delta': np.nan, 'err': np.nan,
                       'sig': np.nan,
                       'n_comp': a['n'], 'n_ref': b['n']}
            continue
        delta = a['median'] - b['median']
        err = float(np.sqrt(a['sem'] ** 2 + b['sem'] ** 2))
        sig = delta / err if err > 0 else np.nan
        out[kj] = {'delta': delta, 'err': err, 'sig': sig,
                   'n_comp': a['n'], 'n_ref': b['n']}
    return out


def _kj_to_array(stats_dict, k_list, j_list, key):
    """Pack one statistic into a (n_k, n_j) array for imshow."""
    Z = np.full((len(k_list), len(j_list)), np.nan)
    for (k, j), s in stats_dict.items():
        if k in k_list and j in j_list:
            Z[k_list.index(k), j_list.index(j)] = s.get(key, np.nan)
    return Z


def plot_kj_heatmap(stats, k_list, j_list, *, value_key='delta',
                    err_key='err', title='', cbar_label='',
                    cmap='RdBu_r', vlim=None, value_fmt='{:+.2f}',
                    err_fmt='±{:.2f}', cell_fontsize=7,
                    show_text=True):
    """Heatmap with k on rows, j on columns, value in colour, and
    optionally `value\n±err` in each cell.

    Returns the figure.
    """
    Z = _kj_to_array(stats, k_list, j_list, value_key)
    Errs = (_kj_to_array(stats, k_list, j_list, err_key)
            if err_key else None)
    if vlim is None:
        finite = Z[np.isfinite(Z)]
        vlim = (float(np.nanpercentile(np.abs(finite), 95))
                if finite.size else 1.0)
        vlim = max(vlim, 1e-4)

    nk, nj = len(k_list), len(j_list)
    fig, ax = plt.subplots(
        figsize=(max(8.0, 0.55 * nj + 1.5),
                 max(2.8, 0.65 * nk + 1.5)),
        layout='constrained')
    im = ax.imshow(Z, cmap=cmap, vmin=-vlim, vmax=vlim,
                   aspect='auto')
    ax.set_xticks(range(nj))
    ax.set_xticklabels([f'Z{j}' for j in j_list], fontsize=8)
    ax.set_yticks(range(nk))
    ax.set_yticklabels([str(k) for k in k_list])
    ax.set_xlabel('Pupil Zernike index j')
    ax.set_ylabel('Field index k')
    cb = plt.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label(cbar_label)

    if show_text:
        for ri in range(nk):
            for ci in range(nj):
                v = Z[ri, ci]
                if not np.isfinite(v):
                    continue
                txt = value_fmt.format(v)
                if Errs is not None and np.isfinite(Errs[ri, ci]):
                    txt += '\n' + err_fmt.format(Errs[ri, ci])
                # Pick black or white text depending on cell brightness.
                color = ('white' if abs(v) > 0.55 * vlim else 'black')
                ax.text(ci, ri, txt, ha='center', va='center',
                        fontsize=cell_fontsize, color=color)

    if title:
        ax.set_title(title, fontsize=12)
    return fig


def to_long_df(stats, bounce_name, ref_label, comp_label,
                ref_stats, comp_stats):
    """Long-format DataFrame for one bounce comparison."""
    rows = []
    for (k, j), d in sorted(stats.items()):
        r = ref_stats.get((k, j), {})
        c = comp_stats.get((k, j), {})
        rows.append({
            'bounce':      bounce_name,
            'reference':   ref_label,
            'comparison':  comp_label,
            'k': int(k), 'j': int(j),
            'n_ref':       int(r.get('n', 0)),
            'ref_median':  float(r.get('median', np.nan)),
            'ref_sigma':   float(r.get('sigma_mad', np.nan)),
            'ref_sem':     float(r.get('sem', np.nan)),
            'n_comp':      int(c.get('n', 0)),
            'comp_median': float(c.get('median', np.nan)),
            'comp_sigma':  float(c.get('sigma_mad', np.nan)),
            'comp_sem':    float(c.get('sem', np.nan)),
            'delta':       float(d.get('delta', np.nan)),
            'delta_err':   float(d.get('err', np.nan)),
            'significance': float(d.get('sig', np.nan)),
        })
    return pd.DataFrame(rows)


def plot_kj_pass_heatmap(deltas, k_list, j_list, *,
                         nsigma_threshold=3.5,
                         delta_threshold_um=0.01,
                         title='',
                         pass_color='#1a936f',
                         cell_fontsize=7,
                         show_text=True):
    """Binary pass/fail heatmap over (k, j).

    A cell passes when BOTH ``|Δ| > delta_threshold_um`` and
    ``|Δ / σ| > nsigma_threshold``.  Passing cells get the single
    ``pass_color`` background and an annotation ``Δ\n±err``; failing
    cells stay white with no annotation.

    Returns the figure.
    """
    from matplotlib.colors import ListedColormap

    nk, nj = len(k_list), len(j_list)
    D = np.full((nk, nj), np.nan)
    E = np.full((nk, nj), np.nan)
    S = np.full((nk, nj), np.nan)
    for (k, j), d in deltas.items():
        if k in k_list and j in j_list:
            ri = k_list.index(k); ci = j_list.index(j)
            D[ri, ci] = d.get('delta', np.nan)
            E[ri, ci] = d.get('err',   np.nan)
            S[ri, ci] = d.get('sig',   np.nan)

    passes = (np.isfinite(D) & np.isfinite(S)
              & (np.abs(D) > delta_threshold_um)
              & (np.abs(S) > nsigma_threshold))
    n_pass = int(passes.sum())
    n_total = int(np.isfinite(D).sum())

    fig, ax = plt.subplots(
        figsize=(max(8.0, 0.55 * nj + 1.5),
                 max(2.8, 0.65 * nk + 1.5)),
        layout='constrained')
    cmap = ListedColormap(['white', pass_color])
    ax.imshow(passes.astype(int), cmap=cmap, vmin=0, vmax=1,
              aspect='auto')
    # Light grid lines between cells for readability.
    ax.set_xticks(np.arange(-0.5, nj, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nk, 1), minor=True)
    ax.grid(which='minor', color='lightgray', linewidth=0.4, alpha=0.6)
    ax.tick_params(which='minor', bottom=False, left=False)

    ax.set_xticks(range(nj))
    ax.set_xticklabels([f'Z{j}' for j in j_list], fontsize=8)
    ax.set_yticks(range(nk))
    ax.set_yticklabels([str(k) for k in k_list])
    ax.set_xlabel('Pupil Zernike index j')
    ax.set_ylabel('Field index k')

    if show_text:
        for ri in range(nk):
            for ci in range(nj):
                if not passes[ri, ci]:
                    continue
                txt = f'{D[ri, ci]:+.3f}\n±{E[ri, ci]:.3f}'
                ax.text(ci, ri, txt, ha='center', va='center',
                        fontsize=cell_fontsize, color='white')

    base = title or 'Significant (k, j) terms'
    ax.set_title(
        f'{base}\nPass: |Δ| > {delta_threshold_um:g} μm  AND  '
        f'|Δ/σ| > {nsigma_threshold:g}    '
        f'({n_pass} / {n_total} cells pass)',
        fontsize=11)
    return fig

<a id='data'></a>
## 4. Data Access

In [ ]:
tables = []
for p in fits_parquet_paths:
    pp = Path(p)
    if not pp.exists():
        raise FileNotFoundError(pp)
    t = QTable.read(str(pp))
    print(f'Loaded {pp.name}: {len(t)} rows, {len(t.colnames)} cols')
    tables.append(t)
fit_table = vstack(tables, metadata_conflicts='silent') if len(tables) > 1 else tables[0]
print(f'\nCombined: {len(fit_table)} rows')

# Drop bad-fit visits.
for bf in (f'{fit_prefix}_bad_fit', 'bad_fit'):
    if bf in fit_table.colnames:
        bad = np.asarray(fit_table[bf]).astype(bool)
        n_bad = int(bad.sum())
        if n_bad > 0:
            print(f'Dropping {n_bad} bad-flagged visits ({bf})')
        fit_table = fit_table[~bad]
        break

# Resolve pupil-Noll j list.
if pupil_j_range is None:
    if 'nollIndices' not in fit_table.colnames:
        raise ValueError(
            "fits parquet has no 'nollIndices' column; set pupil_j_range "
            "explicitly in the parameters cell.")
    iZs = [int(j) for j in np.asarray(fit_table['nollIndices'][0]).tolist()]
else:
    iZs = list(pupil_j_range)
k_list = list(focal_k_range)
print(f'\nPupil Noll j (n={len(iZs)}): {iZs}')
print(f'Focal Noll k (n={len(k_list)}): {k_list}')

# Quick orientation: alt / rotator distribution to confirm the
# bounce-test settings actually live in the data.
if 'alt' in fit_table.colnames:
    alt_deg = _alt_to_deg(fit_table['alt'])
    print(f'\nalt: {np.nanmin(alt_deg):.1f} .. {np.nanmax(alt_deg):.1f} deg, '
          f'mean={np.nanmean(alt_deg):.1f}')
if 'rotator_angle' in fit_table.colnames:
    rot = np.asarray(fit_table['rotator_angle'], dtype=float)
    print(f'rotator_angle: {np.nanmin(rot):.1f} .. {np.nanmax(rot):.1f} deg')

<a id='analysis'></a>
## 5. Bounce Analysis

For each bounce in `bounces`:

1. Build masks for the reference and comparison settings.
2. Compute median, robust RMS (1.4826·MAD), and SEM of the median for
   every (k, j) in each setting.
3. Compute Δ = comparison − reference and propagate the SEM in
   quadrature.

All results land in `bounce_results`, indexed by bounce name and
comparison label.

In [ ]:
bounce_results = {}
long_dfs = []

# Quick orientation: how many visits per science_program?
if 'science_program' in fit_table.colnames:
    from collections import Counter
    sp = np.asarray(fit_table['science_program']).astype(str)
    print('science_program in fit_table:')
    for k, n in sorted(Counter(sp).items(), key=lambda kv: -kv[1]):
        print(f'  {k!r}: {n}')

for b in bounces:
    name = b['name']
    program = b.get('program')
    ref = b['reference']

    ref_kwargs = {k: v for k, v in ref.items() if k != 'label'}
    ref_kwargs.setdefault('program', program)
    ref_mask = filter_visits(fit_table, **ref_kwargs)
    n_ref = int(ref_mask.sum())
    ref_stats = stats_per_kj(fit_table, ref_mask, fit_prefix,
                              k_list, iZs)

    print(f'\n=== {name} ({b.get("description", "")}) ===')
    print(f'  program filter: {program!r}')
    print(f'  reference "{ref["label"]}":  n_visits = {n_ref}')
    print(visit_list_str(fit_table, ref_mask))

    bounce_results[name] = {
        'description': b.get('description', ''),
        'reference_label': ref['label'],
        'reference_stats': ref_stats,
        'reference_n': n_ref,
        'comparisons': {},
    }

    for comp in b['comparisons']:
        comp_kwargs = {k: v for k, v in comp.items() if k != 'label'}
        comp_kwargs.setdefault('program', program)
        comp_mask = filter_visits(fit_table, **comp_kwargs)
        n_comp = int(comp_mask.sum())
        comp_stats = stats_per_kj(fit_table, comp_mask, fit_prefix,
                                   k_list, iZs)
        deltas = diff_stats(comp_stats, ref_stats)
        bounce_results[name]['comparisons'][comp['label']] = {
            'comp_stats': comp_stats,
            'comp_n': n_comp,
            'deltas': deltas,
        }
        print(f'  comparison "{comp["label"]}":  n_visits = {n_comp}')
        print(visit_list_str(fit_table, comp_mask))

        # Quick top-5 Δs by |sig|
        ranked = sorted(
            [(kj, d) for kj, d in deltas.items()
             if np.isfinite(d.get('sig', np.nan))],
            key=lambda kvd: -abs(kvd[1]['sig']))[:5]
        for (k, j), d in ranked:
            print(f'    top |sig|: (k={k}, j={j})  '
                  f'Δ={d["delta"]:+.3f} ± {d["err"]:.3f}  '
                  f'sig={d["sig"]:+.1f}')

        long_dfs.append(to_long_df(
            deltas, name, ref['label'], comp['label'],
            ref_stats, comp_stats))

df_kj = (pd.concat(long_dfs, ignore_index=True)
         if long_dfs else pd.DataFrame())
print(f'\nLong-format table: {len(df_kj)} rows')

<a id='plots'></a>
## 6. Summary Heatmaps

For each `(bounce, comparison)` pair, produce three heatmaps over the
(k, j) grid:

* **Δ DZ heatmap** — colour = `comparison.median - reference.median`,
  cell text = `Δ\n±err`, diverging colormap.
* **Significance heatmap** — colour = `Δ / err`, capped at ±`sig_vlim`.
* **Pass / fail heatmap** — single colour for `(k, j)` cells that pass
  BOTH `|Δ| > pass_delta_threshold_um` and `|Δ/σ| > pass_nsigma_threshold`;
  the rest stay white.  Passing cells are annotated with `Δ\n±err`.

All three are saved to `output_pdf`.

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)

n_pages = 0
with PdfPages(output_pdf) as pdf:
    for name, br in bounce_results.items():
        for comp_label, comp_block in br['comparisons'].items():
            deltas = comp_block['deltas']
            n_ref = br['reference_n']
            n_comp = comp_block['comp_n']
            descr = br['description']

            # 1. Δ heatmap with cell text Δ±err
            fig = plot_kj_heatmap(
                deltas, k_list, iZs,
                value_key='delta', err_key='err',
                title=(f'{name}: Δ DZ_kj = {comp_label} − {br["reference_label"]}\n'
                        f'{descr}  '
                        f'(n_ref={n_ref}, n_comp={n_comp})'),
                cbar_label='Δ DZ [μm]',
                cmap='RdBu_r', vlim=heatmap_vlim_um,
                value_fmt='{:+.3f}', err_fmt='±{:.3f}',
                cell_fontsize=heatmap_cell_fontsize)
            pdf.savefig(fig, bbox_inches='tight')
            if n_pages == 0:
                plt.show()
            plt.close(fig)
            n_pages += 1

            # 2. Significance heatmap (Δ / err)
            fig = plot_kj_heatmap(
                deltas, k_list, iZs,
                value_key='sig', err_key=None,
                title=(f'{name}: significance = Δ / σ  '
                        f'(capped ±{sig_vlim:g}σ)'),
                cbar_label='Δ / σ',
                cmap='RdBu_r', vlim=sig_vlim,
                value_fmt='{:+.1f}', err_fmt='',
                cell_fontsize=heatmap_cell_fontsize)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_pages += 1

            # 3. Binary "real shift" map: passes if |Δ| > delta_th AND
            #    |Δ/σ| > nsigma_th.  Single colour for passing cells.
            fig = plot_kj_pass_heatmap(
                deltas, k_list, iZs,
                nsigma_threshold=pass_nsigma_threshold,
                delta_threshold_um=pass_delta_threshold_um,
                title=f'{name}: significant Δ DZ_kj cells',
                cell_fontsize=heatmap_cell_fontsize)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_pages += 1

print(f'Wrote {n_pages} pages to {output_pdf}')

<a id='output'></a>
## 7. Output Tables

Long-format parquet with one row per `(bounce, comparison, k, j)`,
carrying both the per-setting medians/RMS and the propagated Δ.

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
df_kj.to_parquet(output_table_path)
print(f'Saved: {output_table_path}\n  {len(df_kj)} rows x '
      f'{len(df_kj.columns)} columns')

with pd.option_context('display.max_rows', 50,
                       'display.float_format', '{:+.4f}'.format):
    display(df_kj.sort_values(['bounce', 'comparison', 'j', 'k'])
                  .reset_index(drop=True)
                  .head(50))